In [1]:
import cme.decision_models.confidence_accumulation as ca
import numpy as np
import jax
import numpyro as npy
import numpyro.distributions as dist
import pandas as pd
import seaborn as sns
import scipy.stats as stats


In [51]:
rng = jax.random.key(1)

In [3]:
npy.__version__

'0.15.2'

In [53]:
dist.Beta(2,2).sample(rng, sample_shape=(1,1))

Array([[0.48653431]], dtype=float64)

In [56]:
n_states, start_width, threshold, measurement_prob, delta, mu, sigma, I, J = 51, 9, 5, 1, 0.01, np.asarray([[2]]), np.asarray([[1]]), 10, 5
model_type = "Markov"


In [57]:
Mc, Mw, Mn = ca._get_measurement_matrix(n_states, threshold, prob=measurement_prob, model_type = model_type)
Mc

Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], dtype=float64)

In [58]:
intensity_matrix = ca.get_intensity_matrix(n_states, mu, sigma, model_type=model_type)
phi_0 = ca._get_initial_state(n_states, start_width,model_type=model_type, prior_type="Centered")
intensity_matrix, phi_0

(Array([[[[-1. , -0.5,  0. , ...,  0. ,  0. ,  0. ],
          [ 1. , -1. , -0.5, ...,  0. ,  0. ,  0. ],
          [ 0. ,  1.5, -1. , ...,  0. ,  0. ,  0. ],
          ...,
          [ 0. ,  0. ,  0. , ..., -1. , -0.5,  0. ],
          [ 0. ,  0. ,  0. , ...,  1.5, -1. ,  1. ],
          [ 0. ,  0. ,  0. , ...,  0. ,  1.5, -1. ]]]], dtype=float64),
 Array([[[[0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.11111111],
          [0.11111111],
          [0.11111111],
          [0.11111111],
          [0.11111111],
          [0.11111111],


In [ ]:
t = np.asarray([[4.5]])
phi_t = ca.perform_state_transition(intensity_matrix=intensity_matrix, RT_s=t, RA_s = None, delta=delta, 
Mc = Mc, Mn = Mn, Mw = Mw, phi_0=phi_0, 
transition_type="TIMESTEP", likelihood_type="SINGLE")
phi_t

In [ ]:
S = np.where(dist.MultinomialProbs(phi_t.squeeze()).sample(rng))[0][0]
S

rt = []
ra = []
if S < start_width:
    rt.append(t)
    ra.append(0)
elif S > n_states - start_width:
    rt.append(t)
    ra.append(1)

rt, ra

In [ ]:
stats.uniform(0,1).rvs()

In [ ]:
#stats.multinomial(p=phi_t)

In [ ]:
npy.prng_key()

In [ ]:
rng = jax.random.key(1)
dist.Uniform().sample(npy.prng_key())#, jax.random.uniform(rng)

In [ ]:
def sample_state_1(phi):
    #return np.where(dist.MultinomialProbs(phi_t).sample(npy.prng_key()))[0][0] + 1
    return np.where(stats.multinomial(phi_t).rvs()) + 1

def sample_state(phi):
    phi_cumsum = np.cumsum(phi)
    ran_prob = stats.uniform(0,1).rvs() #dist.Uniform().sample(rng)
    #print(ran_prob)
    S = np.argmax(ran_prob < phi_cumsum)
    return S + 1 # due to 0-based indexing

def random_walk(intensity_matrix, phi_0, delta, T_max, model_type):
    ra = -1
    rt = -1
    

def random_walk(intensity_matrix, phi_0, delta, T_max, model_type):
    ra = -1
    rt = -1
    phi_arr = []
    for t in np.arange(delta, T_max, delta):
        #print(t)
        phi_t = ca.perform_state_transition(intensity_matrix=intensity_matrix, RT_s=np.asarray([[t]]), RA_s = None, delta=delta, 
                                            Mc = Mc, Mn = Mn, Mw = Mw, phi_0=phi_0, 
                                            transition_type="TIMESTEP", likelihood_type="SINGLE")
        if model_type == "Quantum":
            phi_t = np.abs(phi_t)**2
        phi_t = phi_t.squeeze()/phi_t.sum()
        S = sample_state(phi_t) # adding 1 because where returns 0-based indexing
        #print(S, phi_t.sum())
        #states.append(S)
        #phi_arr.append(phi_t)
        if S < threshold:
            ra=0
            rt=t
            break
        elif S > n_states - threshold:
            ra=1
            rt=t
            #print(phi_t, S, t)
            break
    return rt, ra, S, phi_t

In [ ]:
rt_s = []
ra_s = []
for _ in range(1):
    t, ra, S, phi_arr = random_walk(intensity_matrix, phi_0, delta, 100, model_type)
    rt_s.append(t)
    ra_s.append(ra)
rt_s

In [ ]:
from joblib import Parallel, delayed
def get_samples(mu):
    intensity_matrix = ca.get_intensity_matrix(n_states, np.asarray([[mu]]), sigma, model_type=model_type)
    phi_0 = ca._get_initial_state(n_states, start_width,model_type=model_type, prior_type="Uniform")
    
    
    def get_sample():
        t, ra, S, phi_arr = random_walk(intensity_matrix, phi_0, delta, 500, model_type)
        return t, ra, S, phi_arr

    
    ret_arr = Parallel(n_jobs=1)(delayed(get_sample)() for _ in range(10))
    
    for t, ra, S, phi_arr in ret_arr:
        rt_s.append(t)
        ra_s.append(ra)
    return pd.DataFrame({"rt":rt_s, "ra":ra_s, "mu":mu})

In [ ]:
df_x = []
mus = [-1, 1, 2] #stats.norm().rvs(3)
df_x = [get_samples(mu) for mu in mus]


In [ ]:
df_x = pd.concat(df_x)
df_x

In [ ]:
sns.displot(df_x.astype({"mu":"category"}).query("rt > 0.01 "),
            x="rt",
            hue="mu", kind="kde", clip=(0,500))